# Evaluation on trained models

In [ ]:
# mainly loading the environment used for training. Unify everything. (run context)
from dirs import *
from train import *

# Prepare
Manually handle the practical complexity, by loading different trained models and prepare relavant evaluation dataset 

In [ ]:
""" 2025 CLEAR CAE trained on real beam chromox line scan + SGM """

# ========================================
# Load model
# ========================================
import torch
from models.CAE import Autoencoder2D

model_name = "chromox_vae_line_scan_sgm"  
model_path = dirs["models"][model_name] # should point to model.pth
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Autoencoder2D.load_model(
    filepath=model_path,  # should point to model.pth
    device=device,
    eval_mode=True, # inference mode
)
print(show_model_info(model))

# ========================================
# load dataset for inference
# ========================================
from xflow.utils.visualization import plot_image

experiment_name = "CLEAR25"
dataset_sources = ["processed_chromox_cropped"]  # ["processed_dmd", "processed_chromox", "processed_yag", "processed_chromox_laser", "processed_yag_laser"]'

config_manager = ConfigManager(
    load_config(
        f"{experiment_name}.yaml",
        machine=detect_machine(),
        resolve=True,
    )
)
config = config_manager.get()
dataset_dirs = [resolve_dataset_dir(config, src) for src in dataset_sources]
db_rel = config["dataset_structure"]["db"].lstrip("/\\")
db_paths = [d / db_rel for d in dataset_dirs]

eval_provider = SqlProvider(
    sources={"connection": db_paths[0], "sql": config["sql"]["processed_chromox_cropped_random_scan_eval"]}, output_config={'list': "image_path"}
)#.subsample(n_samples=300, seed=config["seed"])

# pad abs path to db saved relative dirs.
for t in config["data"]["transforms"]["torch"]:
    if t.get("name") == "add_parent_dir":
        t.setdefault("params", {})["parent_dir"] = str(dataset_dirs[0])
        break
transforms = build_transforms_from_config(config["data"]["transforms"]["torch"])

# normally leave the pipelines decoupled and untouched so the output interface remains consistent.
eval_dataset = PyTorchPipeline(
    eval_provider,
    transforms
).to_memory_dataset(config["data"]["dataset_ops"])  # testset data do not need thresholding since it is to remove stacking noise?

print("Samples: ", len(eval_provider))
print("Batch: ", len(eval_dataset))
plot_image(next(iter(eval_dataset))[0])
plot_image(next(iter(eval_dataset))[1])

# Inference
get and save intermediate result, fixed downstream (mainly plotting) contract

In [ ]:
"""
MMF speckle reconstruction inference with per-sample image triplets saved to disk.

Ported from the old flat script. The domain-specific logic (triplet figure
saving) lives in TripletSaverHook; the runner handles iteration, device,
eval mode, and batch unpacking.
"""
from xflow.evaluation.runner import (
    run_evaluation,
    TqdmHook,
)
from xflow.extensions.physics.runner import TripletSaverHook

# ----------------------------
# Entry point
# ----------------------------
if __name__ == "__main__":
    # Expected to be defined in the surrounding notebook / script:
    #   model, eval_dataset, device, dirs, model_name

    save_dir = dirs["save"]["model_inference"][model_name]
    saver = TripletSaverHook(save_dir=save_dir)

    ctx = run_evaluation(
        model=model,
        dataset=eval_dataset,
        device=device,
        hooks=[TqdmHook(), saver],
    )

    print(f"batches: {ctx.seen_batches}, samples: {ctx.seen_samples}")

# Visualization
Completely decoupled from above, read only intermediate results with fixed contract, only focus on high quality publication level image, each block is one plot